In [1]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/eserkaraceper/galatasaray-twitter-sentiment-analysis/TWEETS1.CSV


In [2]:
import pandas as pd
from transformers import pipeline

df = pd.read_csv("/kaggle/input/datasets/eserkaraceper/galatasaray-twitter-sentiment-analysis/TWEETS1.CSV")

# Türkçe sentiment modeli
sentiment_model = pipeline(
    "text-classification",
    model="savasy/bert-base-turkish-sentiment-cased",
    truncation=True,
    max_length=512
)

print(f"✅ {len(df)} satır yüklendi, model hazır")

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/442M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

✅ 938 satır yüklendi, model hazır


In [3]:
def analyze(text):
    try:
        result = sentiment_model(str(text)[:512])[0]
        label = result["label"]   # "positive" veya "negative"
        score = result["score"]   # güven skoru (0-1)
        return label, score
    except:
        return "notr", 0.5

print("Skorlanıyor... (938 tweet, ~5-10 dk sürebilir)")

results = df["clean_text"].apply(analyze)
df["label_raw"] = results.apply(lambda x: x[0])
df["confidence"] = results.apply(lambda x: x[1])

print("✅ Bitti")

Skorlanıyor... (938 tweet, ~5-10 dk sürebilir)
✅ Bitti


In [4]:
def map_label(row):
    if row["confidence"] < 0.65:   # düşük güven → nötr
        return "Nötr"
    elif row["label_raw"] == "positive":
        return "Pozitif"
    else:
        return "Negatif"

df["sentiment"] = df.apply(map_label, axis=1)

print(df["sentiment"].value_counts())
print(df["sentiment"].value_counts(normalize=True).mul(100).round(1))
df[["text", "sentiment", "confidence"]].head(10)

sentiment
Negatif    579
Pozitif    257
Nötr       102
Name: count, dtype: int64
sentiment
Negatif    61.7
Pozitif    27.4
Nötr       10.9
Name: proportion, dtype: float64


,text,sentiment,confidence
0,Yav sktrin gidin amk sokacam artık da,Nötr,0.538618
1,bu lesley magassa masalı da 1-2 hafta sorer ge...,Negatif,0.853064
2,#DursunÖzbekParalarNerede,Negatif,0.810933
3,Can -Lesley -Reijnders Can 10 no Lesley 6 Reij...,Pozitif,0.985296
4,Samiyen haberden gör burda gir iyi la 😀😀,Pozitif,0.844890
5,aniden düşen güzel haberler,Nötr,0.575131
6,La siz yalan soyluyonuz inanmiyom hicbirinize,Negatif,0.726683
7,Şu an yaşadıklarımızın rüya olduğunu eylül baş...,Nötr,0.581698
8,Yav şu 23 yaş üstü kontenjanı iyi birini alıp ...,Negatif,0.996077
9,Yalan olduğunu bilsem de inanmak istiyorum,Pozitif,0.953831
